# 04 · Metrics: confusion matrix, ROC and PR curves

After this notebook you can build a confusion matrix, compute precision, recall, F1 and F-beta by hand, explain the accuracy paradox, pick a threshold from a precision–recall curve, say when ROC-AUC misleads and PR-AUC does not, check calibration with a reliability diagram, and choose the right metric for fraud, spam, medical screening, search and LLM guardrails.

**Time:** ~35 min · **Runs:** offline on CPU in < 1 min · **Needs:** [03 · Logistic regression](03_logistic_regression_classification.ipynb)

## Why this matters in interviews

- "How would you evaluate this classifier?" appears in every ML round, and in LLM rounds too: an LLM doing intent routing, moderation or extraction checks is a classifier and is evaluated exactly like this.
- Metric choice is a product decision. Interviewers probe whether you know accuracy lies on imbalanced data and which mistake is the expensive one.
- Guardrails, semantic caches and routers are *threshold* problems: harmful content let through vs over-refusal, wrong cached answers vs hit rate.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `ev17` | How do you evaluate a classification task done by an LLM? |
| `ev25` | How do you evaluate safety and guardrails? |
| `sf10` | What content-safety classifiers would you run, and where? |
| `ev19` | What is Cohen's kappa and why does it matter here? |
| `po10` | What is semantic caching and what threshold would you set? |
| `fo23` | What are logprobs and what can you legitimately use them for? |
| `rg49` | What is the difference between retrieval score and confidence? |
| `sd22` | Design an AI feature where being wrong is expensive: medical, legal or financial. |

**Also asked in classical-ML rounds:** precision vs recall in one sentence each; why F1 is a harmonic mean; what AUC means as a probability; ROC or PR under imbalance; what is calibration and how do you fix it; macro vs micro averaging.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import rankdata
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, average_precision_score, brier_score_loss,
                             classification_report, cohen_kappa_score, confusion_matrix, f1_score, fbeta_score,
                             log_loss, mean_absolute_error, precision_recall_curve, precision_score, recall_score,
                             roc_auc_score, roc_curve, root_mean_squared_error)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(0)
BLUE, ORANGE, AQUA, YELLOW, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8a8984"
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.prop_cycle": plt.cycler(color=[BLUE, ORANGE, AQUA, YELLOW, "#4a3aa7"])})

A fraud-style problem for most of the notebook: 20,000 transactions, about 2% fraudulent, split in half. The model is a plain logistic regression; the numbers, not the model, are the point.

In [ ]:
X, y = make_classification(n_samples=20_000, n_features=10, n_informative=5, n_redundant=3, weights=[0.98],
                           flip_y=0.005, class_sep=1.0, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.5, stratify=y, random_state=0)
model = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_tr, y_tr)
p_te = model.predict_proba(X_te)[:, 1]          # P(fraud) for every test transaction
pred = (p_te >= 0.5).astype(int)                 # what .predict() does
print(f"test rows: {len(y_te)}, fraud cases: {y_te.sum()} ({y_te.mean():.2%})")

## 1. The confusion matrix

**In plain English:** four boxes. Of the things you flagged, some were right (**TP**) and some were false alarms (**FP**). Of the things you let through, some were fine (**TN**) and some you missed (**FN**). Every classification metric is a ratio of these four numbers.

scikit-learn's layout: rows = true class, columns = predicted class, so for labels `[0, 1]` it is `[[TN, FP], [FN, TP]]`.

In [ ]:
def confusion_counts(y_true, y_pred):
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    return tp, fp, fn, tn

tp, fp, fn, tn = confusion_counts(y_te, pred)
print(f"TP={tp}  FP={fp}  FN={fn}  TN={tn}")
assert (confusion_matrix(y_te, pred) == np.array([[tn, fp], [fn, tp]])).all()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay.from_predictions(y_te, pred, display_labels=["legit", "fraud"], cmap="Blues",
                                        colorbar=False, values_format="d", ax=axes[0])
ConfusionMatrixDisplay.from_predictions(y_te, pred, display_labels=["legit", "fraud"], cmap="Blues",
                                        colorbar=False, normalize="true", values_format=".2f", ax=axes[1])
axes[0].set_title("Counts (threshold 0.5)"); axes[1].set_title("Normalised by true class (rows = recall)")
plt.tight_layout(); plt.show()

Read the right-hand panel row by row: the model catches only about half the fraud (bottom row), while almost every legitimate transaction is correctly left alone (top row). The count panel hides this because the top-left box is enormous.

## 2. The accuracy paradox

On 98/2 data, a model that *never* predicts fraud is 98% accurate. Put it next to ours:

In [ ]:
dummy_pred = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr).predict(X_te)
for name, yp in [("always 'legit'", dummy_pred), ("logistic regression", pred)]:
    print(f"{name:>20}: accuracy {accuracy_score(y_te, yp):.4f}   recall {recall_score(y_te, yp):.3f}   "
          f"precision {precision_score(y_te, yp, zero_division=0):.3f}   kappa {cohen_kappa_score(y_te, yp):.3f}")
assert accuracy_score(y_te, dummy_pred) > 0.97 and recall_score(y_te, dummy_pred) == 0

One point of accuracy separates "useless" from "catches half the fraud". Accuracy is dominated by the majority class, so on imbalanced data report recall and precision, or a chance-corrected number.

**Cohen's kappa (`ev19`)** is that chance correction: $\kappa = \frac{p_o - p_e}{1 - p_e}$, observed agreement minus the agreement you would get by luck from the label frequencies alone. The dummy scores exactly 0. The same statistic tells you whether two human annotators, or a human and an LLM judge, genuinely agree.

In [ ]:
def cohen_kappa_np(a, b):
    p_o = np.mean(a == b)
    p_e = sum(np.mean(a == k) * np.mean(b == k) for k in np.union1d(a, b))
    return (p_o - p_e) / (1 - p_e)

assert np.isclose(cohen_kappa_np(y_te, pred), cohen_kappa_score(y_te, pred))
assert np.isclose(cohen_kappa_np(y_te, dummy_pred), 0.0)
print(f"kappa from scratch: {cohen_kappa_np(y_te, pred):.3f} (matches sklearn)")

## 3. Precision, recall, F1 and F-beta, from scratch

| Metric | Formula | Question it answers |
|---|---|---|
| Precision | TP / (TP + FP) | Of what I flagged, how much was right? |
| Recall (sensitivity, TPR) | TP / (TP + FN) | Of what was really there, how much did I catch? |
| Specificity (TNR) | TN / (TN + FP) | Of the negatives, how many did I correctly leave alone? |
| False-positive rate | FP / (FP + TN) = 1 − specificity | How often does a negative trigger an alarm? |
| F1 | 2PR / (P + R) | One number balancing both (harmonic mean) |
| F-beta | (1 + β²)PR / (β²P + R) | β > 1 favours recall, β < 1 favours precision |

In [ ]:
def prf(y_true, y_pred, beta=1.0):
    tp, fp, fn, tn = confusion_counts(y_true, y_pred)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    b2 = beta ** 2
    fbeta = (1 + b2) * precision * recall / (b2 * precision + recall) if precision + recall else 0.0
    return precision, recall, fbeta

P, R, F1 = prf(y_te, pred)
print(f"precision {P:.3f}   recall {R:.3f}   F1 {F1:.3f}")
assert np.isclose(P, precision_score(y_te, pred)) and np.isclose(R, recall_score(y_te, pred))
assert np.isclose(F1, f1_score(y_te, pred))
for beta in (0.5, 2.0):
    assert np.isclose(prf(y_te, pred, beta)[2], fbeta_score(y_te, pred, beta=beta))
    print(f"F{beta:g} = {prf(y_te, pred, beta)[2]:.3f}")

p_, r_ = 1.0, 0.01                            # why harmonic: a model that flags one case and gets it right
print(f"\nP=1.0, R=0.01 -> arithmetic mean {(p_ + r_) / 2:.3f}, F1 {2 * p_ * r_ / (p_ + r_):.3f}")

The harmonic mean is dragged towards the *smaller* number, so you cannot buy a good F1 with perfect precision and no recall. Here precision is high and recall is low, so F0.5 (precision-weighted) beats F2 (recall-weighted).

**More than two classes.** Compute P/R/F1 per class, then average. **Macro** = plain mean over classes (every class counts equally, so rare classes matter). **Micro** = pool all TP/FP/FN first (equals accuracy for single-label problems). **Weighted** = mean weighted by class size. When an LLM classifies intents, report macro-F1 and per-class recall, or the rare intents vanish (`ev17`).

In [ ]:
y3_true = np.repeat([0, 1, 2], [700, 250, 50])            # class 2 is rare
y3_pred = y3_true.copy()
u = rng.random(1000)
y3_pred[(y3_true == 2) & (u < 0.8)] = 0                     # the rare class is mostly missed
y3_pred[(y3_true == 1) & (u < 0.1)] = 0
for avg in ("micro", "macro", "weighted"):
    print(f"{avg:>8} F1: {f1_score(y3_true, y3_pred, average=avg):.3f}")
print(f"accuracy   : {accuracy_score(y3_true, y3_pred):.3f}")
print("per-class recall:", recall_score(y3_true, y3_pred, average=None).round(2))
assert np.isclose(f1_score(y3_true, y3_pred, average="micro"), accuracy_score(y3_true, y3_pred))
assert f1_score(y3_true, y3_pred, average="macro") < f1_score(y3_true, y3_pred, average="micro") - 0.15

## 4. The threshold trade-off

Precision and recall pull against each other as the threshold moves. `precision_recall_curve` returns them at every distinct score; plot them against the threshold to *choose* one.

In [ ]:
prec, rec, thr = precision_recall_curve(y_te, p_te)
f1s = 2 * prec[:-1] * rec[:-1] / np.maximum(prec[:-1] + rec[:-1], 1e-12)
t_f1 = thr[f1s.argmax()]
t_p90 = thr[np.argmax(prec[:-1] >= 0.9)]                   # lowest threshold with precision >= 0.9

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(thr, prec[:-1], lw=2, label="precision")
ax.plot(thr, rec[:-1], lw=2, label="recall")
ax.plot(thr, f1s, lw=2, label="F1")
ax.axvline(0.5, color=GREY, ls="--", lw=1, label="default 0.5")
ax.axvline(t_f1, color="black", lw=1, label=f"best F1 at {t_f1:.2f}")
ax.set(title="Precision, recall and F1 vs threshold (fraud model)", xlabel="threshold on P(fraud)", ylabel="score", xlim=(0, 1))
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5)); plt.show()

for name, t in [("default", 0.5), ("best F1", t_f1), ("precision >= 0.9", t_p90)]:
    P_, R_, F_ = prf(y_te, (p_te >= t).astype(int))
    print(f"{name:>17} (t={t:.3f}): precision {P_:.3f}  recall {R_:.3f}  F1 {F_:.3f}")
assert prf(y_te, (p_te >= t_f1).astype(int))[2] >= prf(y_te, pred)[2]

Lowering the threshold from 0.5 buys recall at the price of precision. The "right" point comes from the product: an analyst team with capacity for 50 alerts a day, a regulator's minimum recall, or the cost ratio from [notebook 03](03_logistic_regression_classification.ipynb). Always pick it on validation data, never on the test set.

### Application: the semantic-cache threshold (`po10`)

A semantic cache returns a stored answer when a new question's embedding is similar enough to an old one. A *false hit* (a different question that happens to look similar, often a negation) serves a wrong answer; a *miss* only costs one LLM call. So: require very high precision on **labelled** question pairs, then take whatever hit rate that allows. The scores below are simulated cosine similarities for 2,000 labelled pairs; with a real embedding model the procedure is identical.

In [ ]:
same = rng.beta(30, 3, size=600)                  # pairs that really are the same question
diff = rng.beta(20, 6, size=1400)                 # different questions, some very similar-looking
sim, is_same = np.r_[same, diff], np.r_[np.ones(600), np.zeros(1400)]

def threshold_for_precision(y_true, scores, target):
    """Lowest threshold whose precision on labelled data reaches the target (maximises recall)."""
    pr, rc, th = precision_recall_curve(y_true, scores)
    ok = np.where(pr[:-1] >= target)[0]
    return th[ok[0]], pr[ok[0]], rc[ok[0]]

for target in (0.90, 0.99):
    t, pr_t, rc_t = threshold_for_precision(is_same, sim, target)
    print(f"precision >= {target:.2f}: threshold {t:.3f}, hit rate on true paraphrases {rc_t:.1%}")
    assert pr_t >= target

Moving from 90% to 99% precision pushes the threshold up and costs hit rate. That is why production caches run high thresholds (0.95 and up) and still add exact-match keys for negation-prone questions.

## 5. The ROC curve and AUC

The ROC curve plots **TPR (recall)** against **FPR** as the threshold sweeps from high to low. It does not depend on any single threshold.

**AUC as a probability:** the area under the ROC curve equals the chance that a random positive gets a higher score than a random negative. That makes it a *ranking* metric, and it is the Mann–Whitney U statistic in disguise. Below: both ways from scratch, checked against scikit-learn.

In [ ]:
def roc_points(y_true, scores):
    order = np.argsort(-scores, kind="stable")          # sweep the threshold from high to low
    hits = y_true[order]
    tpr = np.r_[0, np.cumsum(hits) / hits.sum()]
    fpr = np.r_[0, np.cumsum(1 - hits) / (1 - hits).sum()]
    return fpr, tpr

def auc_by_ranks(y_true, scores):
    ranks = rankdata(scores)                            # average ranks for ties
    n_pos, n_neg = y_true.sum(), (1 - y_true).sum()
    return (ranks[y_true == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)

fpr, tpr = roc_points(y_te, p_te)
auc_trap, auc_rank, auc_sk = np.trapezoid(tpr, fpr), auc_by_ranks(y_te, p_te), roc_auc_score(y_te, p_te)
print(f"AUC: trapezoid {auc_trap:.4f}   ranks {auc_rank:.4f}   sklearn {auc_sk:.4f}")
assert np.isclose(auc_trap, auc_sk, atol=1e-6) and np.isclose(auc_rank, auc_sk)

pos, neg = p_te[y_te == 1], p_te[y_te == 0]
pairs = rng.integers(0, [len(pos), len(neg)], size=(200_000, 2))
print(f"share of random (fraud, legit) pairs ranked correctly: {np.mean(pos[pairs[:, 0]] > neg[pairs[:, 1]]):.4f}")

## 6. The precision–recall curve and average precision

The PR curve plots precision against recall. Its "random model" baseline is **not** a diagonal: it is a horizontal line at the positive rate (2% here). **Average precision (AP)** summarises it: the mean of the precision values at the rank of each true positive.

In [ ]:
def average_precision_np(y_true, scores):
    hits = y_true[np.argsort(-scores, kind="stable")]
    precision_at_k = np.cumsum(hits) / np.arange(1, len(hits) + 1)
    return np.sum(precision_at_k * hits) / hits.sum()

ap_ours, ap_sk = average_precision_np(y_te, p_te), average_precision_score(y_te, p_te)
print(f"average precision: ours {ap_ours:.4f}   sklearn {ap_sk:.4f}")
assert np.isclose(ap_ours, ap_sk)

nb_p = GaussianNB().fit(X_tr, y_tr).predict_proba(X_te)[:, 1]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, s, c in [("logistic regression", p_te, BLUE), ("Gaussian naive Bayes", nb_p, ORANGE)]:
    f_, t_, _ = roc_curve(y_te, s)
    axes[0].plot(f_, t_, lw=2, color=c, label=f"{name} (AUC {roc_auc_score(y_te, s):.3f})")
    p_, r_, _ = precision_recall_curve(y_te, s)
    axes[1].plot(r_, p_, lw=2, color=c, label=f"{name} (AP {average_precision_score(y_te, s):.3f})")
axes[0].plot([0, 1], [0, 1], color=GREY, ls="--", label="random (AUC 0.5)")
axes[1].axhline(y_te.mean(), color=GREY, ls="--", label=f"random (AP = prevalence {y_te.mean():.3f})")
axes[0].set(title="ROC curve", xlabel="false-positive rate", ylabel="true-positive rate (recall)")
axes[1].set(title="Precision-recall curve", xlabel="recall", ylabel="precision")
axes[0].legend(loc="lower right"); axes[1].legend(loc="lower left"); plt.tight_layout(); plt.show()
assert roc_auc_score(y_te, nb_p) > auc_sk and average_precision_score(y_te, nb_p) < ap_sk - 0.1

Notice the ranking of the two models: naive Bayes has the *higher* ROC-AUC but clearly the *lower* AP. ROC rewards pushing positives above the huge mass of easy negatives; PR asks how clean the top of the list is, which is what an analyst working the alert queue experiences.

## 7. Why PR beats ROC under heavy imbalance

Hold the model's quality fixed: fraud scores ~ N(1.5, 1), legit scores ~ N(0, 1). Change only how rare fraud is.

In [ ]:
n_neg = 20_000
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
aucs, aps = [], []
for prevalence, color in [(0.5, BLUE), (0.1, ORANGE), (0.01, AQUA)]:
    n_pos = int(n_neg * prevalence / (1 - prevalence))
    s = np.r_[rng.normal(1.5, 1, n_pos), rng.normal(0, 1, n_neg)]
    yy = np.r_[np.ones(n_pos), np.zeros(n_neg)]
    aucs.append(roc_auc_score(yy, s)); aps.append(average_precision_score(yy, s))
    f_, t_, _ = roc_curve(yy, s); p_, r_, _ = precision_recall_curve(yy, s)
    axes[0].plot(f_, t_, lw=2, color=color, label=f"{prevalence:.0%} positives: AUC {aucs[-1]:.3f}")
    axes[1].plot(r_, p_, lw=2, color=color, label=f"{prevalence:.0%} positives: AP {aps[-1]:.3f}")
axes[0].set(title="ROC: unchanged by prevalence", xlabel="false-positive rate", ylabel="true-positive rate")
axes[1].set(title="PR: collapses as positives get rare", xlabel="recall", ylabel="precision")
for ax in axes:
    ax.legend(loc="lower right" if ax is axes[0] else "upper right")
plt.tight_layout(); plt.show()
assert max(aucs) - min(aucs) < 0.02 and aps[0] - aps[-1] > 0.4

TPR and FPR are each computed *within* one class, so class balance cannot move them. Precision mixes the two classes, so it reacts. At 1% prevalence a "small" 5% false-positive rate means about five false alarms for every real case, and precision falls through the floor while ROC still looks fine. **Rule:** report ROC-AUC for ranking quality when classes are roughly balanced; under heavy imbalance lead with PR-AUC / AP, or precision at the operating point, and always state the prevalence.

## 8. Calibration: does 80% mean 80%?

**In plain English:** a model is *calibrated* if, among everything it scores 0.8, about 80% really are positive. Ranking metrics (AUC, AP) do not care; calibration matters as soon as someone uses the number *as a probability*: expected-cost thresholds, risk scores shown to users, routing on confidence (`fo23`: being sure and being right are different things). A raw similarity score or SVM margin is not a probability at all (`rg49`).

The **reliability diagram** bins predictions and plots the actual positive rate per bin against the mean prediction; perfect calibration is the diagonal. The **Brier score**, mean of $(p - y)^2$, is one number for it (lower is better). Naive Bayes is the classic offender: with correlated features it counts the same evidence several times and becomes overconfident. Balanced data here, so the bins are populated.

In [ ]:
Xc, yc = make_classification(n_samples=6000, n_features=12, n_informative=3, n_redundant=8,
                             class_sep=0.7, flip_y=0.05, random_state=2)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.5, random_state=0)
calib_models = {
    "logistic regression": LogisticRegression(),
    "Gaussian naive Bayes": GaussianNB(),
    "naive Bayes + isotonic calibration": CalibratedClassifierCV(GaussianNB(), method="isotonic", cv=5),
}
probs = {name: m.fit(Xc_tr, yc_tr).predict_proba(Xc_te)[:, 1] for name, m in calib_models.items()}

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot([0, 1], [0, 1], color=GREY, ls="--", label="perfectly calibrated")
for (name, p), c in zip(probs.items(), [BLUE, ORANGE, AQUA]):
    frac_pos, mean_pred = calibration_curve(yc_te, p, n_bins=10)
    axes[0].plot(mean_pred, frac_pos, "o-", lw=2, ms=4, color=c, label=f"{name} (Brier {brier_score_loss(yc_te, p):.3f})")
    axes[1].hist(p, bins=20, range=(0, 1), histtype="step", lw=2, color=c, label=name)
axes[0].set(title="Reliability diagram", xlabel="mean predicted probability", ylabel="actual share of positives")
axes[1].set(title="How spread out the predictions are", xlabel="predicted probability", ylabel="test rows")
axes[0].legend(fontsize=8); axes[1].legend(fontsize=8); plt.tight_layout(); plt.show()

In [ ]:
for name, p in probs.items():
    brier_ours = np.mean((p - yc_te) ** 2)
    assert np.isclose(brier_ours, brier_score_loss(yc_te, p))
    print(f"{name:>35}: Brier {brier_ours:.3f}   log-loss {log_loss(yc_te, p):.3f}   AUC {roc_auc_score(yc_te, p):.3f}")
frac_nb, mean_nb = calibration_curve(yc_te, probs["Gaussian naive Bayes"], n_bins=10)
print()
print(f"naive Bayes, lowest bin : says {mean_nb[0]:.2f}, reality {frac_nb[0]:.2f}")
print(f"naive Bayes, highest bin: says {mean_nb[-1]:.2f}, reality {frac_nb[-1]:.2f}")
b = {k: brier_score_loss(yc_te, p) for k, p in probs.items()}
assert b["naive Bayes + isotonic calibration"] < b["Gaussian naive Bayes"] - 0.03
assert abs(roc_auc_score(yc_te, probs["naive Bayes + isotonic calibration"]) - roc_auc_score(yc_te, probs["Gaussian naive Bayes"])) < 0.01

Naive Bayes piles its predictions up at 0 and 1 (right panel), and its reliability curve is much *flatter* than the diagonal: when it says "almost certainly not" the real rate is over a quarter, and when it says "almost certainly" the real rate is about 80%. That is over-confidence: its probabilities are more extreme than reality. `CalibratedClassifierCV` learns a monotone map from score to probability on held-out folds (`"sigmoid"` = Platt scaling, fine for small data; `"isotonic"` = a step function, needs more data). The Brier score improves while the AUC barely moves: calibration fixes *what the numbers mean*, not *how well the model ranks*.

## 9. `classification_report`: the one-liner

Per-class precision, recall, F1 and **support** (the number of true examples), plus macro and weighted averages. Here at the best-F1 threshold from section 4.

In [ ]:
print(classification_report(y_te, (p_te >= t_f1).astype(int), target_names=["legit", "fraud"], digits=3))

## 10. Regression metrics recap

From [notebook 02](02_linear_regression_and_gradient_descent.ipynb). One wild miss moves RMSE far more than MAE, and percentage errors explode near zero:

In [ ]:
y_true_r = np.array([100.0, 102, 98, 101, 99, 100, 97, 103])
y_good = y_true_r + np.array([1, -2, 1, 0, 2, -1, 1, -2])
y_one_miss = y_good.copy(); y_one_miss[0] = 160                 # one prediction wildly off
for name, yp in [("small errors", y_good), ("plus one wild miss", y_one_miss)]:
    print(f"{name:>19}: MAE {mean_absolute_error(y_true_r, yp):5.2f}   RMSE {root_mean_squared_error(y_true_r, yp):5.2f}")
print("percentage error when the truth is 0.01 and we predict 1.0:", f"{abs(1.0 - 0.01) / 0.01:.0%}")

## 11. Which metric when

| Use case | The expensive mistake | Lead metric | Also report |
|---|---|---|---|
| Fraud detection | missed fraud, but analyst time is limited | recall at a fixed alert budget, precision@k, PR-AUC | money caught, alerts per day |
| Spam filter | a real email sent to spam (FP) | precision / FPR at a high threshold | spam that slips through (recall) |
| Medical screening (`sd22`) | a missed disease (FN) | recall (sensitivity) at acceptable specificity; F2 | precision at the *real* prevalence; the refusal / referral rate |
| Search ranking, RAG retrieval | the right document not in the top k | recall@k, MRR, nDCG ([RAG evaluation](../12_evaluation/03_rag_evaluation_retrieval_and_faithfulness.ipynb)) | latency |
| LLM guardrail classifier (`ev25`, `sf10`) | harm let through AND over-refusal | recall on a red-team set + false-positive rate on benign look-alikes | per-category recall, added latency |
| LLM intent / label classifier (`ev17`) | rare intents ignored | macro-F1, per-class recall, confusion matrix | format-failure rate, cost vs a small baseline |
| Semantic cache (`po10`) | a wrong cached answer | precision ≥ 0.99 on labelled pairs, then hit rate | staleness |
| Probabilities used downstream | over-confidence | Brier score, log-loss, reliability diagram | ECE |
| Balanced classes, equal costs | nothing special | accuracy | confusion matrix |

## Try it yourself

**1. precision@k.** The fraud team can review 100 transactions a day. What share of the model's top-100 scores are real fraud, and what share of all fraud do those 100 reviews catch?

In [ ]:
# Solution — try it yourself first, then run this
k = 100
top_k = np.argsort(-p_te)[:k]
print(f"precision@{k}: {y_te[top_k].mean():.3f}   recall@{k}: {y_te[top_k].sum() / y_te.sum():.3f}")
assert y_te[top_k].mean() > 5 * y_te.mean()          # far better than reviewing at random

**2. A guardrail threshold (`ev25`).** Simulated "unsafe" scores for 300 red-team prompts and 1,000 *benign look-alikes* (security questions, medical questions). Find the lowest threshold whose over-refusal rate (FPR on the benign set) is at most 2%, and report the recall on unsafe prompts.

In [ ]:
# Solution — try it yourself first, then run this
unsafe = rng.beta(6, 2, size=300)
benign = rng.beta(2, 5, size=1000)
scores, is_unsafe = np.r_[unsafe, benign], np.r_[np.ones(300), np.zeros(1000)]
fpr_g, tpr_g, thr_g = roc_curve(is_unsafe, scores)
i = np.where(fpr_g <= 0.02)[0][-1]                   # the most permissive point still within budget
print(f"threshold {thr_g[i]:.3f}: over-refusal {fpr_g[i]:.1%}, unsafe caught {tpr_g[i]:.1%}")
assert fpr_g[i] <= 0.02

**3. Agreement beyond chance (`ev19`).** A human and an LLM judge each label 2,000 answers "good" or "bad". Most answers are good, and the judge says "good" even more often. Compute raw agreement and Cohen's kappa. Why do they tell different stories?

In [ ]:
# Solution — try it yourself first, then run this
human = (rng.random(2000) < 0.85).astype(int)                    # 1 = good
judge = np.where(human == 1, rng.random(2000) < 0.97, rng.random(2000) < 0.60).astype(int)
print(f"raw agreement {np.mean(human == judge):.3f}   Cohen's kappa {cohen_kappa_np(human, judge):.3f}")
assert np.isclose(cohen_kappa_np(human, judge), cohen_kappa_score(human, judge))
# ~87% agreement, but a judge that said "good" to everything would already agree ~85% of the time;
# kappa ~0.4-0.5 ("moderate") is the honest number.

## Say it in an interview

- **30-second answer:** "I start from the confusion matrix and ask which mistake is expensive. Precision is 'of what I flagged, how much was right'; recall is 'of what was there, how much I caught'. I choose the threshold on validation data from the business constraint. For threshold-free comparison I use ROC-AUC when classes are balanced and PR-AUC when positives are rare, because ROC hides false positives among many negatives. If anyone consumes the probabilities, I check calibration with a reliability diagram and the Brier score."
- **Numbers to know:** AUC = P(random positive outscores random negative), 0.5 is random. PR's random baseline = prevalence. F1 is a harmonic mean; F2 weights recall more. Kappa: below 0.4 poor, 0.4 to 0.6 moderate, above 0.8 excellent.
- **Traps:** accuracy on imbalanced data; choosing the threshold on the test set; quoting ROC-AUC 0.95 when precision at the operating point is 10%; comparing AP across datasets with different prevalence; treating naive Bayes, SVM or boosted-tree scores as calibrated probabilities; micro-F1 on imbalanced multiclass (it equals accuracy).
- **LLM translation:** an LLM classifier is evaluated the same way plus a format-failure rate (`ev17`). Guardrails need two numbers: harmful recall and over-refusal rate (`ev25`). LLM-judge agreement with humans is reported as kappa, not raw agreement (`ev19`). A semantic cache threshold is set for precision on labelled pairs (`po10`).
- **Follow-ups:** "How would you pick the threshold?" (cost ratio or capacity, on validation). "Your AUC went up but users are unhappier. How?" (the operating point got worse, or calibration broke).

## Next

- [05 · Overfitting, regularisation and cross-validation](05_overfitting_regularization_cross_validation.ipynb): measuring these metrics reliably.
- [LLM eval metrics](../12_evaluation/01_llm_eval_metrics_em_f1_bleu_rouge.ipynb), [LLM-as-judge and its biases](../12_evaluation/02_llm_as_judge_and_its_biases.ipynb), [RAG evaluation](../12_evaluation/03_rag_evaluation_retrieval_and_faithfulness.ipynb).
- [Guardrails, PII and prompt injection](../16_production/07_guardrails_pii_and_prompt_injection.ipynb) and [caching, exact and semantic](../16_production/03_caching_exact_and_semantic.ipynb).
- Theory: [ml_fundamentals course](../../ml_fundamentals/index.html) (chapter 6), [interview bank](../../ai_interview_prep/index.html).